<a href="https://colab.research.google.com/github/NITYA1J/DSA495/blob/main/DSA495_HW_Week4_Neural_Network_Training_HandWrittenDigits.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Homework Week 4— Neural Network Training with Handwritten Digits

In this homework, you will train a small neural network to classify **8×8 handwritten digit images** using **scikit-learn**.

### Task labels
- **E — Review and Explore:** The existing code is complete. Run it, review it, and feel free to experiment with it.
- **W — Write Your Code:** Write the code needed to complete the step. You may want to follow the provided hints. Please use the specified variable, function, or class names when given, since they may be referenced later.

### Goal
Understand the complete neural-network training workflow:

**Input → Forward Pass → Cross-Entropy Loss → Backpropagation → Weight Update → Evaluation**

## Step 1 — E (Review and Explore): Imports and Dataset

In [ ]:
import warnings
import numpy as np
import matplotlib.pyplot as plt

from sklearn.datasets import load_digits
from sklearn.model_selection import train_test_split
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import accuracy_score, log_loss, ConfusionMatrixDisplay

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
warnings.filterwarnings("ignore", category=UserWarning)

digits = load_digits()

X = digits.data.astype(float) / 16.0
y = digits.target

print("Data shape:", X.shape)
print("Target classes:", np.unique(y))

In [ ]:
sample_id = 8

plt.figure(figsize=(3, 3))
plt.imshow(digits.images[sample_id], cmap="gray")
plt.title(f"True digit = {y[sample_id]}")
plt.axis("off")
plt.show()

## Step 2 — E (Review and Explore): Train/Test Split

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.20,
    random_state=RANDOM_STATE,
    stratify=y,
)

print("Training samples:", len(X_train))
print("Test samples:", len(X_test))

## Step 3 — W (Write Your Code): Build the Neural Network

Create an `MLPClassifier` named **`model`** with this architecture:

**64 inputs → 32 hidden neurons → 10 hidden neurons → 10 output classes**

Use:
- ReLU activation
- Adam optimizer
- learning rate = `0.001`
- batch size = `32`
- `random_state=RANDOM_STATE`

### Hint
Use:

```python
MLPClassifier(
    hidden_layer_sizes=(32, 10),
    activation="relu",
    solver="adam",
    ...
)
```


In [ ]:
# WRITE YOUR CODE HERE

# model = MLPClassifier(
#     ...
# )


## Step 4 — E (Review and Explore): One Training Pass and One Forward Pass

`partial_fit()` performs one training pass. Internally, scikit-learn performs:

**Forward Pass → Cross-Entropy Loss → Backpropagation → Weight Update**

After one pass, we inspect the weight matrices and manually follow one image through the network.

In [ ]:
classes = np.arange(10)

model.partial_fit(
    X_train,
    y_train,
    classes=classes,
)

print("Weight matrix shapes:")
for i, W in enumerate(model.coefs_, start=1):
    print(f"Layer {i}: {W.shape}")

In [ ]:
def relu(x):
    return np.maximum(0, x)

def softmax(z):
    z = z - np.max(z)
    exp_z = np.exp(z)
    return exp_z / exp_z.sum()

x_one = X_test[0]
true_class = y_test[0]

z1 = x_one @ model.coefs_[0] + model.intercepts_[0]
a1 = relu(z1)

z2 = a1 @ model.coefs_[1] + model.intercepts_[1]
a2 = relu(z2)

logits = a2 @ model.coefs_[2] + model.intercepts_[2]
probabilities = softmax(logits)

print("True class:", true_class)
print("Prediction probabilities:", np.round(probabilities, 3))
print("Predicted class:", np.argmax(probabilities))

p_true = probabilities[true_class]
print("Cross-entropy for this example:", round(-np.log(p_true), 4))

## Step 5 — W (Write Your Code): Train for Multiple Epochs

Train the same model for **20 total epochs**.

The first epoch was already completed in Step 4.

Save the loss after each epoch in **`loss_history`**.

### Hint
One additional epoch is:

```python
model.partial_fit(X_train, y_train)
```

The latest training loss is:

```python
model.loss_
```

In [ ]:
EPOCHS = 20
loss_history = [model.loss_]

# WRITE YOUR CODE HERE
# Continue from epoch 2 through epoch 20.
# After each epoch, append model.loss_ to loss_history.


## Step 6 — E (Review and Explore): Plot Training Loss

In [ ]:
plt.figure(figsize=(6, 4))
plt.plot(range(1, len(loss_history) + 1), loss_history, marker="o")
plt.xlabel("Epoch")
plt.ylabel("Cross-Entropy Loss")
plt.title("Training Loss")
plt.grid(alpha=0.25)
plt.show()

## Step 7 — W (Write Your Code): Predict on the Test Set

Using `model` and `X_test`:

- save the predicted classes as **`test_pred`**
- save the class probabilities as **`test_prob`**

### Hint
Use:
- `model.predict(...)`
- `model.predict_proba(...)`

In [ ]:
# WRITE YOUR CODE HERE

# test_pred = ...
# test_prob = ...


## Step 8 — E (Review and Explore): Evaluate the Model

In [ ]:
accuracy = accuracy_score(y_test, test_pred)
test_ce = log_loss(y_test, test_prob)

print(f"Test accuracy: {accuracy:.3f}")
print(f"Test cross-entropy: {test_ce:.3f}")

fig, ax = plt.subplots(figsize=(7, 6))
ConfusionMatrixDisplay.from_predictions(
    y_test,
    test_pred,
    cmap="Blues",
    ax=ax,
)
plt.title("Confusion Matrix")
plt.show()

In [ ]:
for i in range(5):
    pred = test_pred[i]
    true = y_test[i]
    confidence = test_prob[i, pred]

    print(
        f"Example {i}: true={true}, predicted={pred}, "
        f"confidence={confidence:.3f}"
    )

# Final Questions

Answer each question in **2–4 sentences**.

### 1.
Explain the forward pass from the **64 image inputs** to the **10 output probabilities**. What roles do ReLU, logits, softmax, and cross-entropy play?

### 2.
When we call `model.partial_fit(X_train, y_train)`, scikit-learn performs forward propagation, computes the loss, performs backpropagation, and updates the weights. What is the purpose of backpropagation, and what does it mean if the training loss generally decreases?

### 3.
Our neural network predicts one of **10 digit classes**. An LLM predicts the next token from a very large vocabulary. What parts of the training workflow are similar?